# Evaluators

At a high-level, an evaluator judges an invocation of your LLM application against a reference example, and returns an evaluation score.

In LangSmith evaluators, we represent this process as a function that takes in a Run (representing the LLM app invocation) and an Example (representing the data point to evaluate), and returns Feedback (representing the evaluator's score of the LLM app invocation).

![Evaluator](../../images/evaluator.png)

Here is an example of a very simple custom evaluator that compares the output of a model to the expected output in the dataset:

In [ ]:
from langsmith.schemas import Example, Run 

def correct_label(inputs: dict, reference_outputs: dict, outputs: dict) -> dict:
  score = outputs.get("output") == reference_outputs.get("label")
  return {"score": int(score), "key": "correct_label"}

### LLM-as-Judge Evaluation

LLM-as-judge evaluators use LLMs to score system output. To use them, you typically encode the grading rules / criteria in the LLM prompt. They can be reference-free (e.g., check if system output contains offensive content or adheres to specific criteria). Or, they can compare task output to a reference (e.g., check if the output is factually accurate relative to the reference).

Here is an example of how you might define an LLM-as-judge evaluator with structured output

In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. Pydantic contract (Rule #10: lowercase title prevents casing collision on open models)
class Similarity_Score(BaseModel):
    model_config = {"title": "similarity_score"}
    similarity_score: int = Field(
        description="Semantic similarity score between 1 and 10, where 1 means unrelated and 10 means identical."
    )

# 2. Point Judge LLM to Groq ($0)
judge_llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0
).with_structured_output(Similarity_Score)

# 3. The Evaluator Function
def compare_semantic_similarity(inputs: dict, reference_outputs: dict, outputs: dict):
    input_question = inputs["question"]
    reference_response = reference_outputs["output"]
    run_response = outputs["output"]

    messages = [
        {   
            "role": "system",
            "content": (
                "You are a semantic similarity evaluator. Compare the meanings of two responses to a question, "
                "Reference Response and New Response, where the reference is the correct answer, and we are trying to judge if the new response is similar. "
                "Provide a score between 1 and 10, where 1 means completely unrelated, and 10 means identical in meaning."
            ),
        },
        {
            "role": "user", 
            "content": f"Question: {input_question}\nReference Response: {reference_response}\nRun Response: {run_response}"
        }
    ]

    # Invoke structured output judge
    result: Similarity_Score = judge_llm.invoke(messages)
    return {"score": result.similarity_score, "key": "similarity"}

Let's try this out!

NOTE: We purposely made this answer wrong, so we expect to see a low score.

In [ ]:
# From Dataset Example
inputs = {
  "question": "Is LangSmith natively integrated with LangChain?"
}
reference_outputs = {
  "output": "Yes, LangSmith is natively integrated with LangChain, as well as LangGraph."
}


# From Run
outputs = {
  "output": "No, LangSmith is NOT integrated with LangChain."
} 

similarity_score = compare_semantic_similarity(inputs, reference_outputs, outputs)
print(f"Semantic similarity score: {similarity_score}")

Semantic similarity score: {'score': 1, 'key': 'similarity'}


You can also define evaluators using Run and Example directly!

In [7]:
from langsmith.schemas import Run, Example

def compare_semantic_similarity_v2(root_run: Run, example: Example):
    # Safely handle both dictionary fixtures and LangSmith Schema objects
    run_inputs = root_run["inputs"] if isinstance(root_run, dict) else root_run.inputs
    run_outputs = root_run["outputs"] if isinstance(root_run, dict) else root_run.outputs
    example_outputs = example["outputs"] if isinstance(example, dict) else example.outputs

    input_question = run_inputs["question"]
    reference_response = example_outputs["output"]
    run_response = run_outputs["output"]

    messages = [
        {   
            "role": "system",
            "content": (
                "You are a semantic similarity evaluator. Compare the meanings of two responses to a question, "
                "Reference Response and New Response, where the reference is the correct answer, and we are trying to judge if the new response is similar. "
                "Provide a score between 1 and 10, where 1 means completely unrelated, and 10 means identical in meaning."
            ),
        },
        {
            "role": "user", 
            "content": f"Question: {input_question}\n Reference Response: {reference_response}\n Run Response: {run_response}"
        }
    ]

    # Replaces client.beta.chat.completions.parse with Groq-compatible structured output
    result: Similarity_Score = judge_llm.invoke(messages)
    return {"score": result.similarity_score, "key": "similarity"}

In [10]:
sample_run = {
  "name": "Sample Run",
  "inputs": {
    "question": "Is LangSmith natively integrated with LangChain?"
  },
  "outputs": {
    "output": "No, LangSmith is NOT integrated with LangChain."
  },
  "is_root": True,
  "status": "success",
  "extra": {
    "metadata": {
      "key": "value"
    }
  }
}

sample_example = {
  "inputs": {
    "question": "Is LangSmith natively integrated with LangChain?"
  },
  "outputs": {
    "output": "Yes, LangSmith is natively integrated with LangChain, as well as LangGraph."
  },
  "metadata": {
    "dataset_split": [
      "AI generated",
      "base"
    ]
  }
}

similarity_score = compare_semantic_similarity_v2(sample_run, sample_example)
print(f"Semantic similarity score: {similarity_score}")

Semantic similarity score: {'score': 1, 'key': 'similarity'}
